# Étape 4 — Génération DOCX

Transforme `<Livre>_EDIT.txt` en `<Livre>.docx`.

**Aucune IA, aucune clé API, aucun coût.** Uniquement `python-docx` et la convention typographique. Deux exécutions produisent le même document : régénérez autant que vous voulez après avoir ajusté un réglage.

---

**Ce notebook n'est qu'une interface.** Toute la logique vit dans le paquet
`theatre_editor`. On y monte le Drive, on installe les dépendances, on surcharge
éventuellement la configuration, puis on lance l'étape.

**Cette étape est reprenable.** Si Colab coupe, relancez la cellule
d'exécution : le travail déjà validé ne sera pas refait, et vous ne repaierez
aucun appel.

## 1. Dépendances et montage du Drive

In [ ]:
# Installation des dépendances du pipeline.
!pip install -q -U openai pymupdf python-docx

from google.colab import drive

drive.mount("/content/drive")

## 2. Récupération du code

Le dépôt [`elyeskaak/texte_troupe_theatre`](https://github.com/elyeskaak/texte_troupe_theatre)
est **public** : rien à configurer, la cellule suivante suffit. Elle récupère la
dernière version du code à chaque exécution.

> **Si vous repassiez le dépôt en privé**, il faudrait un jeton d'accès :
> GitHub → *Settings* → *Developer settings* → *Personal access tokens* →
> *Fine-grained tokens*, avec **Contents : Read-only** sur ce seul dépôt. Puis
> l'enregistrer dans les Secrets de Colab sous le nom `GITHUB_TOKEN`. La
> cellule le détecte et l'utilise automatiquement — aucune modification à faire.

In [ ]:
# --- Option A : récupération depuis GitHub ------------------------------
DEPOT_COMPTE = "elyeskaak"
DEPOT_NOM = "texte_troupe_theatre"
DOSSIER_PROJET = f"/content/{DEPOT_NOM}"

import os
import subprocess
import sys

# Le jeton est OPTIONNEL : inutile sur un dépôt public, utilisé
# automatiquement s'il est présent. Ainsi la cellule fonctionne dans les deux
# cas, sans qu'il faille se souvenir de la visibilité du dépôt.
jeton = None

try:
    from google.colab import userdata

    jeton = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

# Quand un jeton est utilisé, l'URL le contient : elle ne doit JAMAIS être
# affichée ni figurer dans un message d'erreur. Les sorties de git sont donc
# capturées, jamais relayées.
if jeton:
    url = f"https://{jeton}@github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"
else:
    url = f"https://github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"

if os.path.isdir(DOSSIER_PROJET):
    commande = ["git", "-C", DOSSIER_PROJET, "pull", "--quiet"]
else:
    commande = ["git", "clone", "--quiet", url, DOSSIER_PROJET]

resultat = subprocess.run(commande, capture_output=True, text=True)

if resultat.returncode != 0:
    raise RuntimeError(
        "Récupération du code impossible.\n"
        f"Vérifiez que le dépôt {DEPOT_COMPTE}/{DEPOT_NOM} est accessible.\n"
        "S'il est privé, ajoutez un secret GITHUB_TOKEN dans Colab."
    )

if DOSSIER_PROJET not in sys.path:
    sys.path.insert(0, DOSSIER_PROJET)

print("Code récupéré :", DOSSIER_PROJET)
print("Jeton GitHub  :", "utilisé" if jeton else "non nécessaire (dépôt public)")

In [ ]:
# --- Option B : le dossier theatre_editor/ est sur votre Drive ---------
# Décommentez ces lignes et ajustez le chemin, puis n'exécutez PAS l'option A.

# import sys
# DOSSIER_PROJET = "/content/drive/MyDrive/texte_troupe_theatre"
# if DOSSIER_PROJET not in sys.path:
#     sys.path.insert(0, DOSSIER_PROJET)

## 3. Migration des livres déjà traités

**À lancer une fois**, si vous avez utilisé le pipeline avant le changement de
disposition du Drive.

Le dossier principal ne contient désormais que vos PDF et les DOCX produits ;
tout le travail intermédiaire est rangé dans `temp/<Nom du livre>/`.

Cette cellule déplace les fichiers existants vers la nouvelle disposition. C'est
un simple déplacement : **rien n'est perdu et rien n'est repayé**. Sans elle, vos
transcriptions deviendraient invisibles et seraient refaites.

L'opération ne fait rien si elle a déjà été effectuée.

In [ ]:
from theatre_editor.utils import io

a_migrer = io.livres_a_migrer(config.DOSSIER_DRIVE)

if not a_migrer:
    print("Rien à migrer : la disposition est déjà à jour.")
else:
    for nom in a_migrer:
        print(nom)
        for ligne in io.migrer_livre(nom, config.DOSSIER_DRIVE):
            print("   ", ligne)

    for nom in io.migrer_journaux(config.DOSSIER_DRIVE):
        print("journal :", nom)

## 4. Configuration

`config.py` porte toutes les valeurs par défaut. Les surcharges ci-dessous ne
valent que pour cette session : elles ne modifient pas le fichier.

**Vérifiez le dossier de travail** avant de continuer.

In [ ]:
from pathlib import Path

from theatre_editor import config

# Dossier Drive contenant les PDF et recevant toutes les sorties.
config.DOSSIER_DRIVE = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")

print("Dossier de travail :", config.DOSSIER_DRIVE)
print("Existe             :", config.DOSSIER_DRIVE.is_dir())

## 5. Réglages typographiques

Modifiez librement : cette étape est gratuite et reproductible.

In [ ]:
config.POLICE_TEXTE = "EB Garamond"
config.TAILLE_TEXTE_PT = 15
config.TAILLE_TITRE_ACTE_PT = 20
config.TAILLE_TITRE_SCENE_PT = 18
config.MARGE_CM = 3.0

config.SAUT_DE_PAGE_AVANT_ACTE = True
config.SAUT_DE_PAGE_AVANT_SCENE = False

for cle, definition in config.DEFINITIONS_STYLES.items():
    saut = " + saut de page" if definition["saut_de_page"] else ""
    graisse = "gras" if definition["gras"] else ("italique" if definition["italique"] else "romain")
    print(f"   {definition['nom']:<14} {definition['taille_pt']:>2} pt  "
          f"{definition['alignement']:<9} {graisse}{saut}")

## 6. Table d'inspection de la structure

**À lire avant de générer.** Elle montre comment chaque nom en gras a été
classé — acte, scène, personnage — et signale d'un `⚠` les classements
incertains.

C'est ici qu'on repère un acte pris pour un personnage, plutôt que de le
découvrir à la première page blanche parasite.

In [ ]:
from theatre_editor.utils import blocks, io

for chemin in io.lister_fichiers_edit(config.DOSSIER_DRIVE):
    nom = io.nom_livre_depuis_edit(chemin)
    index = blocks.construire_index_structure(io.lire_texte(chemin))

    print("=" * 72)
    print(nom)
    print("=" * 72)
    print(blocks.rapport_classification(index))
    print()

## 7. Corriger un classement

Si la table ci-dessus se trompe, forcez le classement ici. Écrivez les noms
**en capitales et sans accents**, tels qu'ils apparaissent dans la colonne
`LABEL`.

In [ ]:
# Exemples — décommentez et adaptez :

# config.PERSONNAGES_FORCES = frozenset({"LA VOIX", "LE CHOEUR"})
# config.TITRES_ACTE_FORCES = frozenset({"OUVERTURE"})
# config.TITRES_SCENE_FORCES = frozenset({"ENTRACTE"})

print("Personnages forcés :", sorted(config.PERSONNAGES_FORCES))
print("Actes forcés       :", sorted(config.TITRES_ACTE_FORCES))
print("Scènes forcées     :", sorted(config.TITRES_SCENE_FORCES))

## 8. Génération

In [ ]:
from theatre_editor import docx_export

resultats = docx_export.executer(config.DOSSIER_DRIVE)

## 9. Contrôle du document

Relit le DOCX produit et affiche le style appliqué à chaque paragraphe. Le
meilleur moyen de vérifier qu'actes, scènes et personnages sont bien distingués.

In [ ]:
import docx

for resultat in resultats:
    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)

    if not chemins.docx.exists():
        print(f"{resultat.nom} : aucun document produit.")
        continue

    document = docx.Document(str(chemins.docx))

    print("=" * 72)
    print(f"{resultat.nom} — {len(document.paragraphs)} paragraphes")
    print("=" * 72)

    for paragraphe in document.paragraphs[:40]:
        style = paragraphe.style.name.replace(config.PREFIXE_STYLE, "")
        saut = "  [PAGE NEUVE]" if paragraphe.style.paragraph_format.page_break_before else ""
        print(f"  {style:<14} | {paragraphe.text[:60]}{saut}")

    if len(document.paragraphs) > 40:
        print(f"  … {len(document.paragraphs) - 40} paragraphes de plus")

## 10. Téléchargement

Le document est déjà sur votre Drive. Cette cellule permet de le récupérer
directement sur votre machine.

In [ ]:
from google.colab import files

for resultat in resultats:
    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)

    if chemins.docx.exists():
        files.download(str(chemins.docx))

## 11. À propos de la police

`python-docx` inscrit le **nom** de la police dans le document, il ne
l'incorpore pas. EB Garamond n'a donc pas à être installée dans Colab pour que
la génération réussisse.

En revanche, si elle est absente de la machine qui **ouvre** le fichier, Word
substituera une autre police. Pour un rendu conforme, installez EB Garamond sur
votre poste — elle est gratuite et disponible sur Google Fonts.

## 12. Journal

In [ ]:
# Journal détaillé de l'étape : un enregistrement par appel API, avec sa
# date, son modèle, son response_id, sa durée et sa consommation de jetons.

import json

chemin = config.DOSSIER_DRIVE / config.NOM_JOURNAL.format(etape="docx")

if chemin.exists():
    journal = json.loads(chemin.read_text(encoding="utf-8"))
    print("Dernière exécution :", journal["derniere_execution"])
    print("Configuration      :", json.dumps(journal["configuration"], ensure_ascii=False))
    print()

    for nom, bilan in journal["livres"].items():
        print(f"{nom} : {json.dumps(bilan, ensure_ascii=False)}")

    jetons = sum(
        (appel.get("tokens_entree") or 0) + (appel.get("tokens_sortie") or 0)
        for appel in journal["appels"]
    )
    print()
    print(f"{len(journal['appels'])} appel(s) journalisé(s), {jetons:,} jetons".replace(",", " "))
else:
    print("Aucun journal : l'étape n'a pas encore été lancée.")